# 02 · Construcción del IPBH y variables del modelo

**Proyecto:** Predicción de transiciones de privación básica en hogares colombianos  
**Fuente:** Encuesta Longitudinal Colombiana (ELCA) 2010–2016  
**Autores:** Talia Linares · Camilo Rendón · Nicolás Cubillos · Alejandro Velandia

---

### Lo que hace este notebook

1. Corregir el índice: IPBH (3 dimensiones, denominador correcto)
2. Construir las 14 variables nuevas para el modelo
3. Guardar `base_con_variables.csv` — la usan los notebooks 03 y 04

> ⚠️ **Requisito:** Subir `base_final_modelo.csv` en la primera celda.

## ⚙️ Paso 1 — Subir la base de datos

In [1]:
from google.colab import files
import os

uploaded = files.upload()  # selecciona base_final_modelo.csv

os.makedirs('/content/outputs', exist_ok=True)
print("Archivo subido:", list(uploaded.keys()))

Saving base_final_modelo.csv to base_final_modelo.csv
Archivo subido: ['base_final_modelo.csv']


In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('/content/base_final_modelo.csv')
print(f"Base cargada: {df.shape[0]:,} hogares · {df.shape[1]} variables")
print(f"Períodos: {df['par_ondas'].value_counts().to_dict()}")

Base cargada: 16,957 hogares · 195 variables
Períodos: {'2010_2013': 8897, '2013_2016': 8060}


---
## 1. Diagnóstico del índice original

Verificamos el problema del `ipm_sintetico` antes de corregirlo.

In [3]:
# ============================================================
# CORRECCIÓN SEGÚN RETROALIMENTACIÓN DOCENTE:
# priv_salud ahora incluye EPS + SISBEN
# Un hogar está privado en salud SOLO si no tiene ningún tipo de cobertura
# ============================================================

# Verificar y construir priv_salud corregida
if 'priv_salud' not in df.columns or df['priv_salud'].mean() > 0.30:
    # Si la tasa es >30% probablemente solo usó SISBEN — corregir
    if 'sisben_hogar' in df.columns and 'caja_saludrec' in df.columns:
        df['priv_salud'] = (
            (df['sisben_hogar'] != 1) & (df['caja_saludrec'] != 1)
        ).astype(int)
        print(f"priv_salud CORREGIDA (EPS + SISBEN): {df['priv_salud'].mean()*100:.1f}% privados")
    elif 'sisben_hogar' in df.columns:
        df['priv_salud'] = (df['sisben_hogar'] != 1).astype(int)
        print(f"ADVERTENCIA: usando solo SISBEN — {df['priv_salud'].mean()*100:.1f}% privados")

# Diagnóstico del índice original
# ipm_sintetico puede o no existir dependiendo del notebook 01

if 'ipm_sintetico' in df.columns:
    print("=== Valores únicos del ipm_sintetico ===")
    print(sorted(df['ipm_sintetico'].unique()))
    print()
    print("Con 4 dimensiones esperaríamos: [0, 0.25, 0.5, 0.75, 1.0]  ← esto es lo que hay")
    print("Con 5 dimensiones esperaríamos: [0, 0.2, 0.4, 0.6, 0.8, 1.0]")
    print()
    print(f"priv_ninez valores únicos: {df['priv_ninez'].nunique()}" if 'priv_ninez' in df.columns else "priv_ninez: no encontrada")
    print()
    print("→ priv_ninez es cero en todos los hogares: no aporta al índice")
    print("→ El denominador 4 está mal: solo hay 3 dimensiones con varianza real")
else:
    print("ipm_sintetico no existe en la base — se construirá desde cero como IPBH")
    print()
    # Verificar que las dimensiones necesarias existen
    dims_necesarias = ['priv_educacion', 'priv_salud', 'priv_vivienda']
    for d in dims_necesarias:
        existe = d in df.columns
        print(f"  {'OK' if existe else 'FALTA'}  {d}")

priv_salud CORREGIDA (EPS + SISBEN): 8.6% privados
=== Valores únicos del ipm_sintetico ===
[np.float64(0.0), np.float64(0.25), np.float64(0.5), np.float64(0.75)]

Con 4 dimensiones esperaríamos: [0, 0.25, 0.5, 0.75, 1.0]  ← esto es lo que hay
Con 5 dimensiones esperaríamos: [0, 0.2, 0.4, 0.6, 0.8, 1.0]

priv_ninez valores únicos: 1

→ priv_ninez es cero en todos los hogares: no aporta al índice
→ El denominador 4 está mal: solo hay 3 dimensiones con varianza real


---
## 2. Construcción del IPBH

### ¿Por qué IPBH y no IPM oficial?
El IPM oficial del DANE requiere 15 indicadores en 5 dimensiones. La ELCA no permite construir **niñez** (sin varianza) ni **trabajo** (sin indicadores adecuados). Construimos un índice propio con las 3 dimensiones disponibles.

| Dimensión | Variable | Definición |
|---|---|---|
| Educación | `priv_educacion` | Jefe del hogar sin primaria completa |
| Salud (proxy) | `priv_salud` | Hogar sin registro en el SISBEN |
| Vivienda y servicios | `priv_vivienda` | Carencia en pisos, agua, alcantarillado o hacinamiento |

**Fórmula:** `IPBH = (priv_educacion + priv_salud + priv_vivienda) / 3`

In [9]:
# Verificar y construir dimensiones del IPBH si no existen
# (pueden no estar si el notebook 01 no las generó)

if 'priv_educacion' not in df.columns:
    if 'grado_educ' in df.columns:
        df['priv_educacion'] = (df['grado_educ'] < 3).astype(int)
        print("priv_educacion construida desde grado_educ")
    else:
        df['priv_educacion'] = 0
        print("ADVERTENCIA: grado_educ no encontrada — priv_educacion = 0")

if 'priv_salud' not in df.columns:
    # Privación en salud corregida según retroalimentación de la docente:
    # No basta con no tener SISBEN — hay que verificar que tampoco tiene EPS.
    # Un hogar está privado en salud SOLO si no tiene ningún tipo de cobertura.

    if 'sisben_hogar' in df.columns and 'caja_saludrec' in df.columns:
        df['priv_salud'] = (
            (df['sisben_hogar'] != 1) &
            (df['caja_saludrec'] != 1)
        ).astype(int)

        print(f"priv_salud construida con SISBEN + EPS: {df['priv_salud'].mean()*100:.1f}% privados")

    elif 'sisben_hogar' in df.columns:
        df['priv_salud'] = (df['sisben_hogar'] != 1).astype(int)
        print("ADVERTENCIA: caja_saludrec no disponible — usando solo SISBEN")

    else:
        df['priv_salud'] = 0
        print("ADVERTENCIA: variables de salud no encontradas")
        print("priv_salud construida desde sp_acueducto (proxy SISBEN)")

if 'priv_vivienda' not in df.columns:
    if 'privacion_vivienda' in df.columns:
        df['priv_vivienda'] = df['privacion_vivienda']
        print("priv_vivienda tomada de privacion_vivienda")
    elif all(c in df.columns for c in ['material_pisos','sp_acueducto','sp_alcantarillado']):
        df['priv_vivienda'] = (
            (df['material_pisos'].isin([4,5])).astype(int) +
            (df['sp_acueducto']==2).astype(int) +
            (df['sp_alcantarillado']==2).astype(int)
        ).clip(upper=1)
        print("priv_vivienda construida desde variables de vivienda")
    else:
        df['priv_vivienda'] = 0
        print("ADVERTENCIA: variables de vivienda no encontradas — priv_vivienda = 0")

# Construcción del IPBH
df['ipbh'] = (
    df['priv_educacion'] +
    df['priv_salud'] +
    df['priv_vivienda']
) / 3

df['privado_ipbh'] = (df['ipbh'] >= 1/3).astype(int)

print("\n=== Distribución del IPBH ===")
etiquetas = {0.0:"Sin privaciones (0/3)", 0.333:"Privado en 1 dim (1/3)",
             0.667:"Privado en 2 dims (2/3)", 1.0:"Privado en 3 dims (3/3)"}
for val, cnt in df['ipbh'].round(3).value_counts().sort_index().items():
    pct = cnt/len(df)*100
    label = etiquetas.get(round(val,3), str(val))
    print(f"  {label:<35} {cnt:>6} hogares ({pct:.1f}%)")
print()
print(f"Hogares con IPBH >= 0.33: {df['privado_ipbh'].sum():,} ({df['privado_ipbh'].mean()*100:.1f}%)")


=== Distribución del IPBH ===
  Sin privaciones (0/3)                 8768 hogares (51.7%)
  Privado en 1 dim (1/3)                6260 hogares (36.9%)
  Privado en 2 dims (2/3)               1861 hogares (11.0%)
  Privado en 3 dims (3/3)                 68 hogares (0.4%)

Hogares con IPBH >= 0.33: 8,189 (48.3%)


---
## 3. Construcción de las 14 variables nuevas

### Nota sobre el encoding de la ELCA
Las variables de panel usan esta codificación:

| Par de ondas | Código `No` | Código `Sí` |
|---|---|---|
| 2010–2013 | 1 | 3 |
| 2013–2016 | 0 | 2 |

**Regla:** `valor >= 2` significa "sí tuvo esta condición en el período T".

In [10]:
# -------------------------------------------------------
# 3.1 CHOQUES EXTERNOS
# Capturan si el hogar sufrió un evento adverso
# -------------------------------------------------------
choques = ['inundacion','avalancha','creciente','hundimiento','terremoto','vendaval']
for col in choques:
    df[f'choque_{col}'] = (df[col] >= 2).astype(int)

df['tuvo_choque_climatico'] = (
    df[[f'choque_{c}' for c in choques]].sum(axis=1) > 0
).astype(int)
df['n_tipos_choques'] = df[[f'choque_{c}' for c in choques]].sum(axis=1)

print("=== Choques ===")
print(f"  Hogares con al menos un choque: {df['tuvo_choque_climatico'].mean()*100:.1f}%")

=== Choques ===
  Hogares con al menos un choque: 12.6%


In [11]:
# -------------------------------------------------------
# 3.2 RESILIENCIA FINANCIERA
# Capturan la capacidad del hogar para absorber choques
# -------------------------------------------------------
df['tiene_ahorros_efectivo']  = (df['act_dinero'] >= 2).astype(int)
df['tiene_cesantias']         = (df['act_cesantias'] >= 2).astype(int)
df['tiene_roscas']            = (df['act_roscas'] >= 2).astype(int)
df['tiene_fondos']            = (df['act_fondos'] >= 2).astype(int)
df['acceso_credito_formal']   = (df['credito_financiera'] >= 2).astype(int)
df['apoyo_familiar_finanzas'] = (df['prestamo_familiar'] >= 2).astype(int)

res_cols = ['tiene_ahorros_efectivo','tiene_cesantias','tiene_roscas',
            'tiene_fondos','acceso_credito_formal','apoyo_familiar_finanzas']
df['indice_resiliencia'] = df[res_cols].sum(axis=1)

print("=== Resiliencia financiera ===")
print(f"  Acceso a crédito formal:    {df['acceso_credito_formal'].mean()*100:.1f}%")
print(f"  Índice resiliencia (media): {df['indice_resiliencia'].mean():.2f} / 6")

=== Resiliencia financiera ===
  Acceso a crédito formal:    10.9%
  Índice resiliencia (media): 0.44 / 6


In [12]:
# -------------------------------------------------------
# 3.3 SUBSIDIOS DEL ESTADO
# Capturan si el hogar recibe programas sociales
# -------------------------------------------------------
df['recibe_familias_accion'] = (df['familias_accion'] >= 2).astype(int)
df['recibe_sena']            = (df['sena'] >= 2).astype(int)
df['recibe_red_juntos']      = (df['red_juntos'] >= 2).astype(int)
df['recibe_icbf']            = (df['icbf'] >= 2).astype(int)
df['recibe_adulto_mayor']    = (df['prg_adultomayor'] >= 2).astype(int)
df['recibe_ayuda_alimentos'] = (df['ayu_alimentos'] >= 2).astype(int)

sub_cols = ['recibe_familias_accion','recibe_sena','recibe_red_juntos',
            'recibe_icbf','recibe_adulto_mayor','recibe_ayuda_alimentos']
df['n_programas_estado']    = df[sub_cols].sum(axis=1)
df['recibe_algun_subsidio'] = (df['n_programas_estado'] > 0).astype(int)

print("=== Subsidios del Estado ===")
print(f"  Recibe algún subsidio:      {df['recibe_algun_subsidio'].mean()*100:.1f}%")
print(f"  Recibe Familias en Acción:  {df['recibe_familias_accion'].mean()*100:.1f}%")

=== Subsidios del Estado ===
  Recibe algún subsidio:      48.8%
  Recibe Familias en Acción:  33.9%


In [13]:
# -------------------------------------------------------
# 3.4 ACTIVOS PRODUCTIVOS Y TENENCIA
# -------------------------------------------------------
df['vivienda_propia']     = (df['tenencia_vivienda'] == 1).astype(int)
df['tiene_ingresos_agro'] = (df['agro_ingresos'] >= 2).astype(int)
df['tiene_semovientes']   = (df['semovientes'] > 0).astype(int)
df['tiene_moto']          = (df['motocicletas'] > 0).astype(int)

print("=== Activos productivos ===")
print(f"  Vivienda propia:            {df['vivienda_propia'].mean()*100:.1f}%")
print(f"  Ingresos agropecuarios:     {df['tiene_ingresos_agro'].mean()*100:.1f}%")
print(f"  Tiene semovientes:          {df['tiene_semovientes'].mean()*100:.1f}%")
print()
print("NOTA: Jefatura femenina no disponible en esta base.")
print("Requiere el módulo de personas de la ELCA. Limitación futura.")

=== Activos productivos ===
  Vivienda propia:            23.8%
  Ingresos agropecuarios:     0.1%
  Tiene semovientes:          0.8%

NOTA: Jefatura femenina no disponible en esta base.
Requiere el módulo de personas de la ELCA. Limitación futura.


---
## 4. Resumen de variables construidas

In [14]:
print("=" * 55)
print("RESUMEN DE VARIABLES CONSTRUIDAS")
print("=" * 55)

grupos = {
    'IPBH (índice corregido)': ['ipbh','privado_ipbh'],
    'Choques (2 vars)':        ['tuvo_choque_climatico','n_tipos_choques'],
    'Resiliencia (7 vars)':    ['indice_resiliencia','acceso_credito_formal',
                                'apoyo_familiar_finanzas','tiene_ahorros_efectivo',
                                'tiene_cesantias','tiene_roscas','tiene_fondos'],
    'Subsidios (8 vars)':      ['recibe_familias_accion','recibe_red_juntos',
                                'recibe_icbf','recibe_adulto_mayor',
                                'recibe_ayuda_alimentos','recibe_sena',
                                'n_programas_estado','recibe_algun_subsidio'],
    'Activos (4 vars)':        ['vivienda_propia','tiene_ingresos_agro',
                                'tiene_semovientes','tiene_moto'],
}
total = 0
for grupo, cols in grupos.items():
    print(f"\n  {grupo}")
    for c in cols:
        print(f"    ✓ {c}")
    total += len(cols)
print(f"\n  Total variables nuevas: {total}")
print(f"  Variables base (segunda entrega): 12")
print(f"  TOTAL que entra al modelo: {total + 12 - 2}")  # -2 por ipbh/privado que no van al modelo

RESUMEN DE VARIABLES CONSTRUIDAS

  IPBH (índice corregido)
    ✓ ipbh
    ✓ privado_ipbh

  Choques (2 vars)
    ✓ tuvo_choque_climatico
    ✓ n_tipos_choques

  Resiliencia (7 vars)
    ✓ indice_resiliencia
    ✓ acceso_credito_formal
    ✓ apoyo_familiar_finanzas
    ✓ tiene_ahorros_efectivo
    ✓ tiene_cesantias
    ✓ tiene_roscas
    ✓ tiene_fondos

  Subsidios (8 vars)
    ✓ recibe_familias_accion
    ✓ recibe_red_juntos
    ✓ recibe_icbf
    ✓ recibe_adulto_mayor
    ✓ recibe_ayuda_alimentos
    ✓ recibe_sena
    ✓ n_programas_estado
    ✓ recibe_algun_subsidio

  Activos (4 vars)
    ✓ vivienda_propia
    ✓ tiene_ingresos_agro
    ✓ tiene_semovientes
    ✓ tiene_moto

  Total variables nuevas: 23
  Variables base (segunda entrega): 12
  TOTAL que entra al modelo: 33


---
## 5. Guardar la base consolidada

**Esta es la base que usan los notebooks 03 y 04.** No la vuelvas a construir.

In [15]:
df.to_csv('/content/base_con_variables.csv', index=False)
print(f"✅ base_con_variables.csv guardada")
print(f"   {df.shape[0]:,} hogares · {df.shape[1]} variables")
print()
print("Esta base incluye:")
print("  - Todas las variables originales de la ELCA")
print("  - ipbh y privado_ipbh (índice corregido)")
print("  - 21 variables nuevas construidas en este notebook")
print()
print("➡️  Siguiente paso: notebook 03_eda_COLAB.ipynb")

✅ base_con_variables.csv guardada
   16,957 hogares · 224 variables

Esta base incluye:
  - Todas las variables originales de la ELCA
  - ipbh y privado_ipbh (índice corregido)
  - 21 variables nuevas construidas en este notebook

➡️  Siguiente paso: notebook 03_eda_COLAB.ipynb


---
## 6. Descargar la base para usar en sesiones futuras de Colab

In [16]:
from google.colab import files
files.download('/content/base_con_variables.csv')
print("Descargada: base_con_variables.csv")
print("Guárdala para no tener que correr este notebook de nuevo.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Descargada: base_con_variables.csv
Guárdala para no tener que correr este notebook de nuevo.
